ACll server app5 index7 use korsi eitate


In [ ]:
!pip install open_clip_torch peft flask-cors pyngrok
!pip uninstall torchao -y

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/local_models

project_path = "/content/drive/MyDrive/MrNet-v1/MRNet-v1.0/"
for p in ['sagittal', 'coronal', 'axial']:
    !cp -r "{project_path}lora_{p}_best" /content/local_models/
    !cp "{project_path}attn_{p}_best.pth" /content/local_models/

print("Models copied ✓")

Mounted at /content/drive
Models copied ✓


In [ ]:
%%writefile app.py
# paste the full app.py content here

import os, io, base64, threading, traceback
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.amp import autocast
from PIL import Image
from flask import Flask, request, jsonify
from flask_cors import CORS
import open_clip
from peft import PeftModel
import matplotlib
matplotlib.use('Agg')

app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}})

@app.after_request
def add_ngrok_header(response):
    response.headers['ngrok-skip-browser-warning'] = 'true'
    response.headers['Access-Control-Allow-Origin'] = '*'
    response.headers['Access-Control-Allow-Headers'] = '*'
    response.headers['Access-Control-Allow-Methods'] = 'GET, POST, OPTIONS'
    return response

# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL_BASE_PATH   = os.environ.get("MODEL_BASE_PATH", "/content/local_models")
OPTIMAL_THRESHOLD = 0.2861   # Fixed — from training run
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[ACL-AI] Device    : {DEVICE}")
print(f"[ACL-AI] Models    : {MODEL_BASE_PATH}")
print(f"[ACL-AI] Threshold : {OPTIMAL_THRESHOLD}")

# ── MODEL DEFINITIONS ────────────────────────────────────────────────────────
class GatedAttention(nn.Module):
    def __init__(self, embed_dim=512, hidden_dim=256):
        super().__init__()
        self.attention_V       = nn.Linear(embed_dim, hidden_dim)
        self.attention_U       = nn.Linear(embed_dim, hidden_dim)
        self.attention_weights = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        a_v = torch.tanh(self.attention_V(x))
        a_u = torch.sigmoid(self.attention_U(x))
        w   = self.attention_weights(a_v * a_u)
        A   = torch.softmax(w, dim=0)
        M   = torch.sum(A * x, dim=0, keepdim=True)
        return M, A


class LoRABioMedCLIP(nn.Module):
    def __init__(self, lora_vision_encoder, base_clip_model, text_features):
        super().__init__()
        self.vision_encoder = lora_vision_encoder
        self.text_features  = text_features.detach()
        self.logit_scale    = base_clip_model.logit_scale
        self.attention      = GatedAttention(embed_dim=512, hidden_dim=256)

    def forward(self, image_volume):
        image_features = self.vision_encoder(image_volume)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        patient_image_feature, attn_weights = self.attention(image_features)
        patient_image_feature = patient_image_feature / patient_image_feature.norm(dim=-1, keepdim=True)
        logit_scale = self.logit_scale.exp()
        logits = logit_scale * patient_image_feature @ self.text_features.t()
        return logits, attn_weights


# ── GLOBAL MODEL CACHE ────────────────────────────────────────────────────────
clip_model    = None
preprocess    = None
tokenizer     = None
_model_loaded = False
_load_lock    = threading.Lock()


def load_base_model():
    global clip_model, preprocess, tokenizer, _model_loaded
    with _load_lock:
        if _model_loaded:
            return
        print("[ACL-AI] Loading BioMedCLIP base model…")
        model_name = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        clip_model, _, preprocess = open_clip.create_model_and_transforms(model_name)
        clip_model   = clip_model.to(DEVICE)
        tokenizer    = open_clip.get_tokenizer(model_name)
        _model_loaded = True
        print("[ACL-AI] Base model loaded ✓")


# ── HELPERS ───────────────────────────────────────────────────────────────────
def get_ensembled_text_features(plane_name):
    prompts_healthy = [
        f"A normal {plane_name} MRI slice of the knee with an intact anterior cruciate ligament.",
        f"Healthy knee MRI showing a continuous ACL."
    ]
    prompts_tear = [
        f"A {plane_name} MRI slice of the knee showing a complete tear of the anterior cruciate ligament.",
        f"Ruptured anterior cruciate ligament with discontinuity."
    ]
    clip_model.eval()
    with torch.no_grad():
        tok_healthy  = tokenizer(prompts_healthy).to(DEVICE)
        feat_healthy = clip_model.encode_text(tok_healthy).mean(dim=0, keepdim=True)
        tok_tear     = tokenizer(prompts_tear).to(DEVICE)
        feat_tear    = clip_model.encode_text(tok_tear).mean(dim=0, keepdim=True)
        text_features = torch.cat([feat_healthy, feat_tear], dim=0)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features


def load_patient_volume_from_bytes(npy_bytes):
    vol = np.load(io.BytesIO(npy_bytes))
    vol = (vol - vol.min()) / (vol.max() - vol.min() + 1e-6)
    processed_slices = []
    for i in range(vol.shape[0]):
        slice_img = cv2.resize(vol[i], (224, 224))
        slice_rgb = np.stack(((slice_img * 255).astype(np.uint8),) * 3, axis=-1)
        processed_slices.append(preprocess(Image.fromarray(slice_rgb)))
    return torch.stack(processed_slices), vol


def generate_better_heatmap(single_slice, model, text_feats):
    """Exact copy of notebook Cell 8 — high-quality saliency."""
    model.vision_encoder.zero_grad()

    input_slice    = single_slice.clone().detach().requires_grad_(True)
    image_features = model.vision_encoder(input_slice)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

    tear_feature = text_feats[1:2]
    logit_scale  = model.logit_scale.exp()
    similarity   = (logit_scale * image_features @ tear_feature.t()).squeeze()
    similarity.backward()

    grad     = input_slice.grad.data.squeeze().cpu().numpy()
    saliency = np.sqrt(np.mean(np.square(np.maximum(grad, 0)), axis=0))
    saliency = cv2.GaussianBlur(saliency, (31, 31), 10)
    saliency = np.power(saliency, 2)

    v_min, v_max = saliency.min(), np.percentile(saliency, 99)
    if v_max > v_min:
        saliency = np.clip((saliency - v_min) / (v_max - v_min), 0, 1)
    else:
        saliency = np.zeros_like(saliency)

    return cv2.applyColorMap(np.uint8(255 * saliency), cv2.COLORMAP_JET)


def get_plane_analysis(npy_bytes, plane_name):
    """Exact logic of notebook Cell 8 updated get_plane_analysis()."""
    text_feats   = get_ensembled_text_features(plane_name)
    adapter_path = os.path.join(MODEL_BASE_PATH, f"lora_{plane_name}_best")
    attn_path    = os.path.join(MODEL_BASE_PATH, f"attn_{plane_name}_best.pth")

    if hasattr(clip_model.visual, "peft_config"):
        delattr(clip_model.visual, "peft_config")

    lora_vision = PeftModel.from_pretrained(clip_model.visual, adapter_path)
    model       = LoRABioMedCLIP(lora_vision, clip_model, text_feats).to(DEVICE)
    model.attention.load_state_dict(torch.load(attn_path, map_location=DEVICE))
    model.eval()

    img_tensor, vol_raw = load_patient_volume_from_bytes(npy_bytes)
    img_tensor_device   = img_tensor.to(DEVICE)

    with torch.no_grad():
        with autocast('cuda'):
            logits, attn_weights = model(img_tensor_device)
            prob_tear            = logits.softmax(dim=-1)[0, 1].item()
            target_slice_idx     = torch.argmax(attn_weights).item()

    single_slice  = img_tensor_device[target_slice_idx:target_slice_idx + 1]
    heatmap_color = generate_better_heatmap(single_slice, model, text_feats)

    clip_model.visual = lora_vision.unload()
    return prob_tear, target_slice_idx, heatmap_color, vol_raw


def arr_to_b64png(arr_rgb):
    buf = io.BytesIO()
    Image.fromarray(arr_rgb).save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()


def build_visual_panel(vol_raw, idx, heat_bgr):
    slice_img_resized = cv2.resize(vol_raw[idx], (224, 224))
    slice_rgb         = np.stack(((slice_img_resized * 255).astype(np.uint8),) * 3, axis=-1)
    heat_rgb          = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)
    overlay           = cv2.addWeighted(slice_rgb, 0.5, heat_rgb, 0.5, 0)
    return {
        'original': arr_to_b64png(slice_rgb),
        'heatmap':  arr_to_b64png(heat_rgb),
        'overlay':  arr_to_b64png(overlay),
    }


def build_clinical_report(patient_id, ensemble_prob, threshold,
                           s_prob, s_idx, c_prob, c_idx, a_prob, a_idx):
    """
    Exact logic from notebook Cell 9 final version —
    calculates weighted contributions and identifies primary driver.
    """
    pred_status = "ACL Tear" if ensemble_prob >= threshold else "Intact/Healthy"

    s_contrib = 0.6 * s_prob
    c_contrib = 0.3 * c_prob
    a_contrib = 0.1 * a_prob

    contributions  = {"Sagittal": s_contrib, "Coronal": c_contrib, "Axial": a_contrib}
    primary_driver = max(contributions, key=contributions.get)

    is_tear = ensemble_prob >= threshold
    interpretation = (
        f"Based on the BioMedCLIP multi-planar ensemble, the patient exhibits a "
        f"{'high' if is_tear else 'low'} probability of an ACL injury. "
        f"While the Axial plane showed a high raw probability, the {primary_driver} "
        f"plane was the primary driver of this diagnosis due to its higher clinical weight "
        f"and strong feature alignment. The attention network successfully isolated critical "
        f"slices (e.g., Sagittal Slice #{s_idx}) showing significant signal disruption."
    )

    return {
        "patient_id":      patient_id,
        "pred_status":     pred_status,
        "ensemble_pct":    round(ensemble_prob * 100, 1),
        "sagittal_pct":    round(s_prob * 100, 1),
        "sagittal_contrib": round(s_contrib * 100, 1),
        "sagittal_slice":  s_idx,
        "coronal_pct":     round(c_prob * 100, 1),
        "coronal_contrib":  round(c_contrib * 100, 1),
        "coronal_slice":   c_idx,
        "axial_pct":       round(a_prob * 100, 1),
        "axial_contrib":   round(a_contrib * 100, 1),
        "axial_slice":     a_idx,
        "primary_driver":  primary_driver,
        "interpretation":  interpretation,
    }


# ── ROUTES ────────────────────────────────────────────────────────────────────
@app.route('/health', methods=['GET', 'OPTIONS'])
def health():
    return jsonify({'status': 'ok', 'device': str(DEVICE)})


@app.route('/analyze', methods=['POST', 'OPTIONS'])
def analyze():
    if request.method == 'OPTIONS':
        return jsonify({}), 200

    load_base_model()

    for plane in ('sagittal', 'coronal', 'axial'):
        if plane not in request.files:
            return jsonify({'error': f'Missing file: {plane}'}), 400

    patient_id = request.form.get('patient_id', 'Unknown')
    sag_bytes  = request.files['sagittal'].read()
    cor_bytes  = request.files['coronal'].read()
    ax_bytes   = request.files['axial'].read()

    try:
        print(f"[ACL-AI] ── Analyzing patient: {patient_id} ──────────────")
        s_prob, s_idx, s_heat, s_vol = get_plane_analysis(sag_bytes, 'sagittal')
        print(f"[ACL-AI]  Sagittal : {s_prob*100:.1f}%  (top slice #{s_idx})")
        c_prob, c_idx, c_heat, c_vol = get_plane_analysis(cor_bytes, 'coronal')
        print(f"[ACL-AI]  Coronal  : {c_prob*100:.1f}%  (top slice #{c_idx})")
        a_prob, a_idx, a_heat, a_vol = get_plane_analysis(ax_bytes, 'axial')
        print(f"[ACL-AI]  Axial    : {a_prob*100:.1f}%  (top slice #{a_idx})")
    except FileNotFoundError as e:
        return jsonify({'error': f'Model file not found: {e}. Did the setup cell run?'}), 500
    except Exception as e:
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

    ensemble  = (0.6 * s_prob) + (0.3 * c_prob) + (0.1 * a_prob)
    threshold = OPTIMAL_THRESHOLD
    diagnosis = "ACL Tear Detected" if ensemble >= threshold else "Healthy / Intact ACL"
    print(f"[ACL-AI]  Ensemble : {ensemble*100:.1f}%  →  {diagnosis}")

    report = build_clinical_report(
        patient_id, ensemble, threshold,
        s_prob, s_idx, c_prob, c_idx, a_prob, a_idx
    )

    return jsonify({
        'diagnosis':  diagnosis,
        'confidence': round(ensemble, 6),
        'report':     report,
        'patient_id': patient_id,
        'planes': {
            'sagittal': {'probability': round(s_prob * 100, 1), 'top_slice': s_idx,
                         **build_visual_panel(s_vol, s_idx, s_heat)},
            'coronal':  {'probability': round(c_prob * 100, 1), 'top_slice': c_idx,
                         **build_visual_panel(c_vol, c_idx, c_heat)},
            'axial':    {'probability': round(a_prob * 100, 1), 'top_slice': a_idx,
                         **build_visual_panel(a_vol, a_idx, a_heat)},
        }
    })


# ── ENTRY POINT ───────────────────────────────────────────────────────────────
if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False, threaded=False)


Writing app.py


In [ ]:
import threading, time, subprocess
from pyngrok import ngrok

# Start Flask in background
def run_flask():
    subprocess.run(["python", "app.py"])

t = threading.Thread(target=run_flask, daemon=True)
t.start()

# Wait for Flask to bind to port 5000
time.sleep(4)
print("Flask started ✓")

# Set your ngrok auth token (free account at dashboard.ngrok.com)
ngrok.set_auth_token("3DTudsXOILvOgALXoUMOBniIDQx_5u2hGfzVuJt2522DCjWyA")

# Open tunnel
tunnel = ngrok.connect(5000, bind_tls=True)
print("─" * 50)
print("SERVER URL:", tunnel.public_url)
print("─" * 50)
print("Copy this URL → paste into index.html → BACKEND_URL")

Flask started ✓
──────────────────────────────────────────────────
SERVER URL: https://populate-legible-shiny.ngrok-free.dev
──────────────────────────────────────────────────
Copy this URL → paste into index.html → BACKEND_URL
